In [2]:
# =============================================================================
# E7 — Sentence-transformers + PCA + CatBoost (Colab T4 version)
# =============================================================================
# КАК ИСПОЛЬЗОВАТЬ:
# 1. Создай новый Colab notebook
# 2. Runtime → Change runtime type → T4 GPU → Save
# 3. Загрузи e7_colab_input.zip через левую панель Files (drag & drop)
# 4. Скопируй ВЕСЬ этот код в первую ячейку и нажми Run
# 5. Жди ~15-25 минут
# 6. В конце скачай e7_results.zip из Files panel
# =============================================================================

import os, sys, json, time, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 0. Setup: распаковать input + установить зависимости
# -----------------------------------------------------------------------------
print("=" * 70)
print("STEP 0: Setup")
print("=" * 70)

# Проверка GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: no GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("   Continuing on CPU but it will be slow (~30-60 min).")

# Распаковать input
if not os.path.exists("e7_colab_input.zip"):
    raise FileNotFoundError(
        "e7_colab_input.zip not found in /content/. "
        "Загрузи zip через левую панель Files."
    )

if not os.path.exists("train_canon.csv"):
    print("Unzipping e7_colab_input.zip...")
    with zipfile.ZipFile("e7_colab_input.zip", "r") as z:
        z.extractall(".")
    print("Files extracted:")
    for f in sorted(os.listdir(".")):
        if not f.startswith("."):
            size = os.path.getsize(f) / 1024 if os.path.isfile(f) else 0
            print(f"  {f} ({size:.1f} KB)" if os.path.isfile(f) else f"  {f}/")

# Установка sentence-transformers + catboost
print("\nInstalling dependencies (silent)...")
os.system("pip install -q sentence-transformers catboost")
print("Done.")


# -----------------------------------------------------------------------------
# 1. Load data
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 1: Load data")
print("=" * 70)

train_df = pd.read_csv("train_canon.csv")
val_df = pd.read_csv("val_canon.csv")
test_df = pd.read_csv("test_canon.csv")

with open("feature_cols_e5.json") as f:
    feature_cols_e5 = json.load(f)

print(f"train: {train_df.shape}")
print(f"val:   {val_df.shape}")
print(f"test:  {test_df.shape}")
print(f"feature_cols_e5: {len(feature_cols_e5)} cols")
assert len(train_df) == 135626
assert len(val_df) == 27156
assert len(test_df) == 34416
assert len(feature_cols_e5) == 42
print("✅ Sizes match expected (135626/27156/34416, 42 features)")

# Reference probas (для bootstrap comparisons)
ref = {}
for name in ["e0_canon", "e5", "e6_spw15"]:
    path = f"reference_probas/test_proba_{name}.npy"
    if os.path.exists(path):
        ref[name] = np.load(path)
        print(f"  reference proba {name}: shape={ref[name].shape}")
    else:
        print(f"  ⚠️ {path} missing — bootstrap comparisons with {name} will be skipped")

y_test = np.load("reference_probas/y_test_canon.npy") if os.path.exists("reference_probas/y_test_canon.npy") \
    else test_df["resolution"].values
print(f"y_test: shape={y_test.shape}, sum={y_test.sum()}")


# -----------------------------------------------------------------------------
# 2. Sentence-transformers embeddings
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 2: Embeddings (multilingual-e5-small)")
print("=" * 70)

from sentence_transformers import SentenceTransformer

model_name = "intfloat/multilingual-e5-small"
print(f"Loading {model_name}...")
t0 = time.time()
model = SentenceTransformer(model_name)
print(f"Loaded in {time.time()-t0:.1f}s, device={model.device}")


def build_text(row):
    """Склейка 4 текстовых полей с passage: префиксом для multilingual-e5."""
    parts = []
    for col in ["name_rus", "description", "brand_name", "CommercialTypeName4"]:
        v = row[col]
        if pd.notna(v) and str(v).strip() and str(v) != "nan":
            parts.append(str(v))
    return "passage: " + " ".join(parts) if parts else "passage: empty"


# Encode каждый split отдельно — экономнее по памяти
def encode_split(df, name, batch_size=128):
    texts = df.apply(build_text, axis=1).tolist()
    t0 = time.time()
    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    dt = time.time() - t0
    print(f"  {name}: {embs.shape}  in {dt:.1f}s ({len(texts)/dt:.0f} texts/sec)")
    assert not np.isnan(embs).any(), f"NaN in {name} embeddings!"
    return embs


t0 = time.time()
emb_train = encode_split(train_df, "train", batch_size=128)
emb_val = encode_split(val_df, "val", batch_size=128)
emb_test = encode_split(test_df, "test", batch_size=128)
total_encode_time = time.time() - t0
print(f"\nTotal encoding time: {total_encode_time:.1f}s = {total_encode_time/60:.1f} min")

# Сохранить эмбеддинги
np.save("emb_train.npy", emb_train)
np.save("emb_val.npy", emb_val)
np.save("emb_test.npy", emb_test)


# -----------------------------------------------------------------------------
# 3. PCA-25 (fit на train, transform на val/test)
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 3: PCA-25")
print("=" * 70)

from sklearn.decomposition import PCA

pca = PCA(n_components=25, random_state=42)
text_pca_train = pca.fit_transform(emb_train)
text_pca_val = pca.transform(emb_val)
text_pca_test = pca.transform(emb_test)

explained = pca.explained_variance_ratio_.sum()
print(f"PCA shapes: train={text_pca_train.shape}, val={text_pca_val.shape}, test={text_pca_test.shape}")
print(f"Explained variance @ 25 components: {explained:.4f} ({explained*100:.1f}%)")
print(f"Reference E5 (CLIP-512 → PCA-25) had explained variance: 57.9%")


# -----------------------------------------------------------------------------
# 4. CatBoost training (mirror of E5 from Cell 27)
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 4: CatBoost training (mirror of E5)")
print("=" * 70)

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

pca_cols = [f"text_pca_{i}" for i in range(25)]

text_pca_train_df = pd.DataFrame(text_pca_train, columns=pca_cols, index=train_df.index)
text_pca_val_df = pd.DataFrame(text_pca_val, columns=pca_cols, index=val_df.index)
text_pca_test_df = pd.DataFrame(text_pca_test, columns=pca_cols, index=test_df.index)


def make_e7_X(df_part, pca_df):
    return pd.concat(
        [df_part[feature_cols_e5].reset_index(drop=True),
         pca_df.reset_index(drop=True)],
        axis=1,
    )


X_train_e7 = make_e7_X(train_df, text_pca_train_df)
X_val_e7 = make_e7_X(val_df, text_pca_val_df)
X_test_e7 = make_e7_X(test_df, text_pca_test_df)

cats_e7 = X_train_e7.select_dtypes(include=["object", "string", "category"]).columns.tolist()
for col in cats_e7:
    X_train_e7[col] = X_train_e7[col].fillna("missing").astype(str)
    X_val_e7[col] = X_val_e7[col].fillna("missing").astype(str)
    X_test_e7[col] = X_test_e7[col].fillna("missing").astype(str)

print(f"X_train_e7: {X_train_e7.shape}")
print(f"X_val_e7:   {X_val_e7.shape}")
print(f"X_test_e7:  {X_test_e7.shape}")
print(f"Categorical features ({len(cats_e7)}): {cats_e7}")

# Mirror E5 config from Cell 27 of fintech_experiment.ipynb
print("\nTraining CatBoost (E5 mirror config: iter=500, lr=0.05, depth=6, no spw)...")
t0 = time.time()
m_e7 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=100,
    eval_metric="AUC",
)
m_e7.fit(
    X_train_e7,
    train_df["resolution"],
    cat_features=cats_e7,
    eval_set=(X_val_e7, val_df["resolution"]),
    use_best_model=True,
)
catboost_time = time.time() - t0
print(f"\nTraining done in {catboost_time:.1f}s")
print(f"best_iteration_: {m_e7.best_iteration_} / 500")
if m_e7.best_iteration_ >= 498:
    print("⚠️  best_iteration ≥ 498 — model didn't converge (same as E5). Logging anyway.")

proba_e7 = m_e7.predict_proba(X_test_e7)[:, 1]
np.save("test_proba_e7.npy", proba_e7)


# -----------------------------------------------------------------------------
# 5. Metrics
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 5: Metrics")
print("=" * 70)


def compute_oracle_metrics(y_true, y_score):
    roc = roc_auc_score(y_true, y_score)
    pr_auc = average_precision_score(y_true, y_score)
    p, r, _ = precision_recall_curve(y_true, y_score)
    mask = p[:-1] >= 0.9
    r_at_p90 = r[:-1][mask].max() if mask.any() else 0.0
    return roc, pr_auc, r_at_p90


roc_e7, pr_e7, r90_e7 = compute_oracle_metrics(y_test, proba_e7)
print(f"E7: ROC={roc_e7:.4f}  PR-AUC={pr_e7:.4f}  R@P90={r90_e7:.4f}")

# Reference values from results_table_v2.csv
print("\nReference (from results_table_v2.csv):")
print("  E0_canon: ROC=0.9208  PR-AUC=0.6587  R@P90=0.1721")
print("  E5:       ROC=0.9232  PR-AUC=0.6724  R@P90=0.2956")
print("  E6_spw15: ROC=0.9192  PR-AUC=0.6541  R@P90=0.2078")


# -----------------------------------------------------------------------------
# 6. Bootstrap CI (paired, n_iter=1000)
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 6: Bootstrap CI (paired, n_iter=1000, seed=42)")
print("=" * 70)


def bootstrap_independent(y_true, y_score, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    pr_aucs, recalls = [], []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt, ys = y_true[idx], y_score[idx]
        if yt.sum() == 0:
            continue
        pr_aucs.append(average_precision_score(yt, ys))
        p, r, _ = precision_recall_curve(yt, ys)
        mask = p[:-1] >= 0.9
        recalls.append(r[mask].max() if mask.any() else 0.0)
    return {
        "pr_auc_mean": np.mean(pr_aucs),
        "pr_auc_ci_low": np.quantile(pr_aucs, 0.025),
        "pr_auc_ci_high": np.quantile(pr_aucs, 0.975),
        "recall_p90_mean": np.mean(recalls),
        "recall_p90_ci_low": np.quantile(recalls, 0.025),
        "recall_p90_ci_high": np.quantile(recalls, 0.975),
        "n_iter_effective": len(pr_aucs),
    }


def bootstrap_paired_diff(y_true, score_a, score_b, n_iter=1000, seed=42):
    """Paired bootstrap: на одном ресэмпле считаем обе модели, затем их разность."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs_pr, diffs_r = [], []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if yt.sum() == 0:
            continue
        sa, sb = score_a[idx], score_b[idx]
        # PR-AUC delta
        diffs_pr.append(average_precision_score(yt, sb) - average_precision_score(yt, sa))
        # R@P90 delta
        p_a, r_a, _ = precision_recall_curve(yt, sa)
        p_b, r_b, _ = precision_recall_curve(yt, sb)
        mask_a = p_a[:-1] >= 0.9
        mask_b = p_b[:-1] >= 0.9
        ra = r_a[mask_a].max() if mask_a.any() else 0.0
        rb = r_b[mask_b].max() if mask_b.any() else 0.0
        diffs_r.append(rb - ra)
    diffs_pr = np.array(diffs_pr)
    diffs_r = np.array(diffs_r)
    return {
        "pr_auc_delta": np.mean(diffs_pr),
        "pr_auc_ci_low": np.quantile(diffs_pr, 0.025),
        "pr_auc_ci_high": np.quantile(diffs_pr, 0.975),
        "pr_auc_significant": (np.quantile(diffs_pr, 0.025) > 0) or (np.quantile(diffs_pr, 0.975) < 0),
        "r_p90_delta": np.mean(diffs_r),
        "r_p90_ci_low": np.quantile(diffs_r, 0.025),
        "r_p90_ci_high": np.quantile(diffs_r, 0.975),
        "r_p90_significant": (np.quantile(diffs_r, 0.025) > 0) or (np.quantile(diffs_r, 0.975) < 0),
    }


# Independent CI for E7
print("Computing independent CI for E7...")
ci_e7 = bootstrap_independent(y_test, proba_e7, n_iter=1000, seed=42)
print(f"  E7 PR-AUC = {ci_e7['pr_auc_mean']:.4f} "
      f"[{ci_e7['pr_auc_ci_low']:.4f}, {ci_e7['pr_auc_ci_high']:.4f}]")
print(f"  E7 R@P90  = {ci_e7['recall_p90_mean']:.4f} "
      f"[{ci_e7['recall_p90_ci_low']:.4f}, {ci_e7['recall_p90_ci_high']:.4f}]")

# Paired comparisons
pairwise = []
for ref_name in ["e0_canon", "e5", "e6_spw15"]:
    if ref_name not in ref:
        print(f"  ⚠️ Skipping E7 vs {ref_name}: reference proba missing")
        continue
    print(f"\nPaired bootstrap: E7 vs {ref_name}...")
    cmp = bootstrap_paired_diff(y_test, ref[ref_name], proba_e7, n_iter=1000, seed=42)
    print(f"  delta R@P90 = {cmp['r_p90_delta']:+.4f}  "
          f"CI95=[{cmp['r_p90_ci_low']:+.4f}, {cmp['r_p90_ci_high']:+.4f}]  "
          f"significant={cmp['r_p90_significant']}")
    print(f"  delta PR-AUC = {cmp['pr_auc_delta']:+.4f}  "
          f"CI95=[{cmp['pr_auc_ci_low']:+.4f}, {cmp['pr_auc_ci_high']:+.4f}]  "
          f"significant={cmp['pr_auc_significant']}")
    pairwise.append({
        "comparison": f"E7 vs {ref_name}",
        "metric": "recall_p90",
        "delta": cmp["r_p90_delta"],
        "ci_low": cmp["r_p90_ci_low"],
        "ci_high": cmp["r_p90_ci_high"],
        "significant": cmp["r_p90_significant"],
    })
    pairwise.append({
        "comparison": f"E7 vs {ref_name}",
        "metric": "pr_auc",
        "delta": cmp["pr_auc_delta"],
        "ci_low": cmp["pr_auc_ci_low"],
        "ci_high": cmp["pr_auc_ci_high"],
        "significant": cmp["pr_auc_significant"],
    })


# -----------------------------------------------------------------------------
# 7. Save all results
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 7: Save artifacts")
print("=" * 70)

os.makedirs("e7_results", exist_ok=True)

# Bootstrap CI E7 (independent)
pd.DataFrame([{
    "experiment": "E7_text_e5small_PCA25",
    **ci_e7,
}]).to_csv("e7_results/e7_bootstrap_ci.csv", index=False)

# Pairwise comparisons
pd.DataFrame(pairwise).to_csv("e7_results/e7_bootstrap_pairwise.csv", index=False)

# E7 row для results_table_v2
pd.DataFrame([{
    "experiment": "E7_text_e5small_PCA25",
    "train_size": 135626,
    "roc_auc": roc_e7,
    "pr_auc": pr_e7,
    "recall_at_p90_oracle": r90_e7,
    "delta_r_at_p90_vs_e0_canon": r90_e7 - 0.1721,
}]).to_csv("e7_results/e7_results_row.csv", index=False)

# Probas + PCA для будущих анализов
shutil.copy("test_proba_e7.npy", "e7_results/test_proba_e7.npy")
np.save("e7_results/text_pca25_train.npy", text_pca_train)
np.save("e7_results/text_pca25_val.npy", text_pca_val)
np.save("e7_results/text_pca25_test.npy", text_pca_test)

# Summary text
summary = f"""# E7 Summary

## Setup
- Model: {model_name} (384-dim)
- Texts: 4 fields concatenated with `passage: ` prefix
- Encoding time: {total_encode_time:.1f}s ({total_encode_time/60:.1f} min)
- PCA explained variance @ 25: {explained:.4f}

## Test metrics (canon split, n=34416)
- ROC-AUC: {roc_e7:.4f}
- PR-AUC:  {pr_e7:.4f}
- R@P90:   {r90_e7:.4f}
- best_iteration: {m_e7.best_iteration_} / 500

## Reference (results_table_v2.csv)
- E0_canon: 0.1721
- E5:       0.2956
- E6_spw15: 0.2078

## Independent CI for E7
- PR-AUC: [{ci_e7['pr_auc_ci_low']:.4f}, {ci_e7['pr_auc_ci_high']:.4f}]
- R@P90:  [{ci_e7['recall_p90_ci_low']:.4f}, {ci_e7['recall_p90_ci_high']:.4f}]

## Paired comparisons (recall_p90)
"""
for row in pairwise:
    if row["metric"] == "recall_p90":
        summary += (f"- {row['comparison']}: delta={row['delta']:+.4f}  "
                    f"CI95=[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  "
                    f"significant={row['significant']}\n")

summary += "\n## Interpretation scaffolding (to be decided by author, not by Claude)\n"
if r90_e7 > 0.30:
    summary += "→ E7 > E5: text modality stronger than visual. Rewrite §3.5 main narrative.\n"
elif r90_e7 > 0.27:
    summary += "→ E7 ≈ E5: symmetric multimodal contribution. Strongest narrative possible.\n"
elif r90_e7 > 0.20:
    summary += "→ E7 between E0 and E5: text helps, but visual dominates. Close text gap.\n"
elif r90_e7 > 0.17:
    summary += "→ E7 ≈ E0_canon: sentence-tr does NOT recover text modality vs CatBoost target encoding.\n"
else:
    summary += "→ E7 < E0_canon: unexpected — sentence-tr hurts baseline. Diagnostics required.\n"

with open("e7_results/E7_SUMMARY.md", "w") as f:
    f.write(summary)

print("Saved to e7_results/:")
for f in sorted(os.listdir("e7_results")):
    size_kb = os.path.getsize(f"e7_results/{f}") / 1024
    print(f"  {f} ({size_kb:.1f} KB)")


# -----------------------------------------------------------------------------
# 8. Zip everything for download
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("STEP 8: Create download zip")
print("=" * 70)

with zipfile.ZipFile("e7_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("e7_results"):
        for file in files:
            full = os.path.join(root, file)
            arc = os.path.relpath(full, "e7_results")
            zf.write(full, arc)

zip_size_mb = os.path.getsize("e7_results.zip") / (1024 * 1024)
print(f"\n✅ DONE: e7_results.zip ({zip_size_mb:.1f} MB)")
print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(summary)
print()
print("Скачай e7_results.zip из левой панели Files (правый клик → Download).")

STEP 0: Setup
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Unzipping e7_colab_input.zip...
Files extracted:
  README.md (0.8 KB)
  e7_colab_input.zip (46237.3 KB)
  feature_cols_e5.json (1.0 KB)
  reference_probas/
  sample_data/
  test_canon.csv (30220.1 KB)
  train_canon.csv (129385.4 KB)
  val_canon.csv (26697.9 KB)

Installing dependencies (silent)...
Done.

STEP 1: Load data
train: (135626, 45)
val:   (27156, 45)
test:  (34416, 45)
feature_cols_e5: 42 cols
✅ Sizes match expected (135626/27156/34416, 42 features)
  reference proba e0_canon: shape=(34416,)
  reference proba e5: shape=(34416,)
  reference proba e6_spw15: shape=(34416,)
y_test: shape=(34416,), sum=2040

STEP 2: Embeddings (multilingual-e5-small)
Loading intfloat/multilingual-e5-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/213 [00:00<?, ?it/s]

  val: (27156, 384)  in 136.2s (199 texts/sec)


Batches:   0%|          | 0/269 [00:00<?, ?it/s]

  test: (34416, 384)  in 156.4s (220 texts/sec)

Total encoding time: 995.7s = 16.6 min

STEP 3: PCA-25
PCA shapes: train=(135626, 25), val=(27156, 25), test=(34416, 25)
Explained variance @ 25 components: 0.4351 (43.5%)
Reference E5 (CLIP-512 → PCA-25) had explained variance: 57.9%

STEP 4: CatBoost training (mirror of E5)
X_train_e7: (135626, 67)
X_val_e7:   (27156, 67)
X_test_e7:  (34416, 67)
Categorical features (4): ['CommercialTypeName4', 'brand_name', 'description', 'name_rus']

Training CatBoost (E5 mirror config: iter=500, lr=0.05, depth=6, no spw)...
0:	test: 0.9048417	best: 0.9048417 (0)	total: 457ms	remaining: 3m 47s
100:	test: 0.9475984	best: 0.9475984 (100)	total: 19s	remaining: 1m 15s
200:	test: 0.9529585	best: 0.9529635 (199)	total: 43.9s	remaining: 1m 5s
300:	test: 0.9560227	best: 0.9560227 (300)	total: 1m 9s	remaining: 45.7s
400:	test: 0.9576915	best: 0.9576915 (400)	total: 1m 33s	remaining: 23.1s
499:	test: 0.9590617	best: 0.9590617 (499)	total: 1m 57s	remaining: 0us

IndexError: boolean index did not match indexed array along axis 0; size of axis is 21533 but size of corresponding boolean axis is 21532

In [3]:
# =============================================================================
# RESCUE: bootstrap CI + save + zip (после падения Step 6)
# =============================================================================
import os, zipfile, shutil
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

# proba_e7, y_test, ref - уже в памяти от предыдущей ячейки
print(f"E7 proba shape: {proba_e7.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"reference probas: {list(ref.keys())}")


def _r_at_p90(y_true, y_score):
    """Безопасная версия recall@precision>=0.9 — учитывает разную длину массивов."""
    p, r, _ = precision_recall_curve(y_true, y_score)
    # precision и recall имеют одну и ту же длину (n+1), thresholds — n
    # Берём пары (p, r) где p >= 0.9 и возвращаем max recall
    mask = p >= 0.9
    if not mask.any():
        return 0.0
    return r[mask].max()


def bootstrap_independent(y_true, y_score, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    pr_aucs, recalls = [], []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        ys = y_score[idx]
        if yt.sum() == 0:
            continue
        pr_aucs.append(average_precision_score(yt, ys))
        recalls.append(_r_at_p90(yt, ys))
    return {
        "pr_auc_mean": float(np.mean(pr_aucs)),
        "pr_auc_ci_low": float(np.quantile(pr_aucs, 0.025)),
        "pr_auc_ci_high": float(np.quantile(pr_aucs, 0.975)),
        "recall_p90_mean": float(np.mean(recalls)),
        "recall_p90_ci_low": float(np.quantile(recalls, 0.025)),
        "recall_p90_ci_high": float(np.quantile(recalls, 0.975)),
        "n_iter_effective": len(pr_aucs),
    }


def bootstrap_paired_diff(y_true, score_a, score_b, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs_pr, diffs_r = [], []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if yt.sum() == 0:
            continue
        sa, sb = score_a[idx], score_b[idx]
        diffs_pr.append(average_precision_score(yt, sb) - average_precision_score(yt, sa))
        diffs_r.append(_r_at_p90(yt, sb) - _r_at_p90(yt, sa))
    diffs_pr = np.array(diffs_pr)
    diffs_r = np.array(diffs_r)
    return {
        "pr_auc_delta": float(np.mean(diffs_pr)),
        "pr_auc_ci_low": float(np.quantile(diffs_pr, 0.025)),
        "pr_auc_ci_high": float(np.quantile(diffs_pr, 0.975)),
        "pr_auc_significant": bool((np.quantile(diffs_pr, 0.025) > 0) or (np.quantile(diffs_pr, 0.975) < 0)),
        "r_p90_delta": float(np.mean(diffs_r)),
        "r_p90_ci_low": float(np.quantile(diffs_r, 0.025)),
        "r_p90_ci_high": float(np.quantile(diffs_r, 0.975)),
        "r_p90_significant": bool((np.quantile(diffs_r, 0.025) > 0) or (np.quantile(diffs_r, 0.975) < 0)),
    }


# ---- Independent CI for E7
print("\nIndependent CI for E7...")
ci_e7 = bootstrap_independent(y_test, proba_e7, n_iter=1000, seed=42)
print(f"  E7 PR-AUC = {ci_e7['pr_auc_mean']:.4f}  "
      f"[{ci_e7['pr_auc_ci_low']:.4f}, {ci_e7['pr_auc_ci_high']:.4f}]")
print(f"  E7 R@P90  = {ci_e7['recall_p90_mean']:.4f}  "
      f"[{ci_e7['recall_p90_ci_low']:.4f}, {ci_e7['recall_p90_ci_high']:.4f}]")

# ---- Paired comparisons
pairwise = []
for ref_name in ["e0_canon", "e5", "e6_spw15"]:
    if ref_name not in ref:
        continue
    print(f"\nPaired bootstrap: E7 vs {ref_name}...")
    cmp = bootstrap_paired_diff(y_test, ref[ref_name], proba_e7, n_iter=1000, seed=42)
    print(f"  delta R@P90 = {cmp['r_p90_delta']:+.4f}  "
          f"CI95=[{cmp['r_p90_ci_low']:+.4f}, {cmp['r_p90_ci_high']:+.4f}]  "
          f"significant={cmp['r_p90_significant']}")
    print(f"  delta PR-AUC = {cmp['pr_auc_delta']:+.4f}  "
          f"CI95=[{cmp['pr_auc_ci_low']:+.4f}, {cmp['pr_auc_ci_high']:+.4f}]  "
          f"significant={cmp['pr_auc_significant']}")
    for metric_name, prefix in [("recall_p90", "r_p90"), ("pr_auc", "pr_auc")]:
        pairwise.append({
            "comparison": f"E7 vs {ref_name}",
            "metric": metric_name,
            "delta": cmp[f"{prefix}_delta"],
            "ci_low": cmp[f"{prefix}_ci_low"],
            "ci_high": cmp[f"{prefix}_ci_high"],
            "significant": cmp[f"{prefix}_significant"],
        })


# ---- Recompute E7 main metrics (на всякий случай)
from sklearn.metrics import roc_auc_score
roc_e7 = float(roc_auc_score(y_test, proba_e7))
pr_e7 = float(average_precision_score(y_test, proba_e7))
r90_e7 = float(_r_at_p90(y_test, proba_e7))


# ---- Save artifacts
os.makedirs("e7_results", exist_ok=True)
pd.DataFrame([{"experiment": "E7_text_e5small_PCA25", **ci_e7}]).to_csv(
    "e7_results/e7_bootstrap_ci.csv", index=False)
pd.DataFrame(pairwise).to_csv("e7_results/e7_bootstrap_pairwise.csv", index=False)
pd.DataFrame([{
    "experiment": "E7_text_e5small_PCA25",
    "train_size": 135626,
    "roc_auc": roc_e7,
    "pr_auc": pr_e7,
    "recall_at_p90_oracle": r90_e7,
    "delta_r_at_p90_vs_e0_canon": r90_e7 - 0.1721,
}]).to_csv("e7_results/e7_results_row.csv", index=False)
shutil.copy("test_proba_e7.npy", "e7_results/test_proba_e7.npy")

# Summary
summary = f"""# E7 Summary

## Setup
- Model: intfloat/multilingual-e5-small (384-dim)
- PCA explained variance @ 25: {0.4351:.4f}

## Test metrics (canon split, n=34416)
- ROC-AUC: {roc_e7:.4f}
- PR-AUC:  {pr_e7:.4f}
- R@P90:   {r90_e7:.4f}
- best_iteration: 499 / 500

## Reference (results_table_v2.csv)
- E0_canon: 0.1721
- E5:       0.2956
- E6_spw15: 0.2078

## Independent CI for E7
- PR-AUC: [{ci_e7['pr_auc_ci_low']:.4f}, {ci_e7['pr_auc_ci_high']:.4f}]
- R@P90:  [{ci_e7['recall_p90_ci_low']:.4f}, {ci_e7['recall_p90_ci_high']:.4f}]

## Paired comparisons (recall_p90)
"""
for row in pairwise:
    if row["metric"] == "recall_p90":
        summary += (f"- {row['comparison']}: delta={row['delta']:+.4f}  "
                    f"CI95=[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  "
                    f"significant={row['significant']}\n")

with open("e7_results/E7_SUMMARY.md", "w") as f:
    f.write(summary)


# ---- Zip everything
with zipfile.ZipFile("e7_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("e7_results"):
        for file in files:
            full = os.path.join(root, file)
            arc = os.path.relpath(full, "e7_results")
            zf.write(full, arc)

print(f"\n{'=' * 70}")
print(f"✅ DONE. e7_results.zip is ready in Files panel.")
print(f"{'=' * 70}")
print(summary)


E7 proba shape: (34416,)
y_test shape: (34416,)
reference probas: ['e0_canon', 'e5', 'e6_spw15']

Independent CI for E7...
  E7 PR-AUC = 0.6659  [0.6454, 0.6868]
  E7 R@P90  = 0.3197  [0.2557, 0.3572]

Paired bootstrap: E7 vs e0_canon...
  delta R@P90 = +0.1332  CI95=[+0.0363, +0.2389]  significant=True
  delta PR-AUC = +0.0073  CI95=[-0.0027, +0.0170]  significant=False

Paired bootstrap: E7 vs e5...
  delta R@P90 = +0.0308  CI95=[-0.0136, +0.0746]  significant=False
  delta PR-AUC = -0.0064  CI95=[-0.0122, +0.0001]  significant=False

Paired bootstrap: E7 vs e6_spw15...
  delta R@P90 = +0.1075  CI95=[+0.0306, +0.1741]  significant=True
  delta PR-AUC = +0.0118  CI95=[+0.0042, +0.0187]  significant=True

✅ DONE. e7_results.zip is ready in Files panel.
# E7 Summary

## Setup
- Model: intfloat/multilingual-e5-small (384-dim)
- PCA explained variance @ 25: 0.4351

## Test metrics (canon split, n=34416)
- ROC-AUC: 0.9285
- PR-AUC:  0.6660
- R@P90:   0.3299
- best_iteration: 499 / 500

## 